In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/同花顺')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:6974,第一支股票:000001


,Id,Code,Date,Open,High,Low,Close,Amount,Volume
0,1,000001,2014-01-02 00:00:00,12.12,12.30,12.05,12.23,596223740.0,48991089.0
1,2,000001,2014-01-03 00:00:00,12.15,12.16,11.78,11.93,656631300.0,55111484.0
2,3,000001,2014-01-06 00:00:00,11.89,12.00,11.50,11.67,679280380.0,58211823.0
3,4,000001,2014-01-07 00:00:00,11.53,11.76,11.51,11.63,393977580.0,33840749.0
4,5,000001,2014-01-08 00:00:00,11.64,11.95,11.53,11.76,538436170.0,45776816.0


In [2]:
dt['sma10']=talib.SMA(dt['Close'], 10)

In [4]:
# 我要看昨天收盘价低于均线但今天收盘价高于均线的
dt['preClose'] = dt['Close'].shift()
dt['sma10']=talib.SMA(dt['Close'], 10)
dt['preSma10']=dt['sma10'].shift()
# 接下来多少天的涨幅
for days in [5,10,20]:
    dt[f'next{days}Close']=dt['Close'].shift(-days) # 多少日之后的收盘价
    dt[f'rate{days}'] = ((dt[f'next{days}Close']-dt['Close'])/dt['Close'])*100 # 判断涨幅的

dt2 = dt.loc[(dt['preClose']<dt['preSma10']) & (dt['Close']>dt['sma10']), :]
dt2.head()    

,Id,Code,Date,Open,High,Low,Close,Amount,Volume,sma10,preClose,preSma10,next5Close,rate5,next10Close,rate10,next20Close,rate20
14,15,000001,2014-01-22 00:00:00,11.36,11.86,11.35,11.79,7.235232e+08,61908449.0,11.625,11.36,11.622,11.56,-1.950806,11.94,1.272265,11.11,-5.767600
19,20,000001,2014-01-29 00:00:00,11.54,11.64,11.50,11.56,3.615753e+08,31210441.0,11.543,11.50,11.554,11.94,3.287197,12.00,3.806228,10.70,-7.439446
22,23,000001,2014-02-10 00:00:00,11.38,11.59,11.35,11.57,6.700087e+08,58122625.0,11.531,11.38,11.504,11.97,3.457217,11.16,-3.543647,10.31,-10.890233
29,30,000001,2014-02-19 00:00:00,11.61,12.13,11.61,12.00,1.177763e+09,98817485.0,11.822,11.66,11.778,11.11,-7.416667,10.70,-10.833333,10.24,-14.666667
51,52,000001,2014-03-21 00:00:00,10.07,10.94,10.03,10.79,1.316755e+09,125181194.0,10.376,10.10,10.377,10.78,-0.092678,10.79,0.000000,10.69,-0.926784


In [6]:
for days in [5,10,20]:
    m = dt[f'rate{days}'].mean()
    print(f'{days},{m}')

5,0.0901757672871568
10,0.16579309340990545
20,0.30872464004958405


In [7]:
for days in [5,10,20]:
    print(f'{days},avg:{dt[f'rate{days}'].mean()},min:{dt[f'rate{days}'].min()}, max:{dt[f'rate{days}'].max()}')

5,avg:0.0901757672871568,min:-22.901385493072528, max:25.316455696202528
10,avg:0.16579309340990545,min:-26.323987538940802, max:41.28014842300558
20,avg:0.30872464004958405,min:-25.039619651347056, max:44.496707431796786


In [8]:
# 这里进行全部股票
dts = []
for i in tqdm(range(len(codes))):
    code = codes[i]
    dt = datasource.getData(code, '2024-01-01')
    dt['preClose'] = dt['Close'].shift()
    dt['sma10']=talib.SMA(dt['Close'], 10)
    dt['preSma10']=dt['sma10'].shift()
    # 接下来多少天的涨幅
    for days in [5,10,20]:
        dt[f'next{days}Close']=dt['Close'].shift(-days) # 多少日之后的收盘价
        dt[f'rate{days}'] = ((dt[f'next{days}Close']-dt['Close'])/dt['Close'])*100 # 判断涨幅的
    
    dt2 = dt.loc[(dt['preClose']<dt['preSma10']) & (dt['Close']>dt['sma10']), :]
    if len(dt2) > 0:
        dts.append(dt2)
    

100%|█████████████████████████████████████████████████████████████████████████████| 6974/6974 [01:08<00:00, 101.63it/s]


In [19]:
dt_all = pd.concat(dts)
counts1 = len(dt_all)
for days in [5,10,20]:
    # 计算一下上涨的比例
    counts2 = len(dt_all.loc[dt_all[f'rate{days}']>0,:])
    rate = counts2/counts1*100
    print(f'{days},avg:{dt_all[f'rate{days}'].mean():.2f}%,min:{dt_all[f'rate{days}'].min():.2f}%, max:{dt_all[f'rate{days}'].max():.2f}%，上涨几率:{rate:.2f}%')

5,avg:0.95%,min:-57.88%, max:256.96%，上涨几率:51.28%
10,avg:1.20%,min:-65.95%, max:307.96%，上涨几率:50.12%
20,avg:2.81%,min:-79.06%, max:461.18%，上涨几率:50.93%


看起来突破10日均线后，半数的比例是上涨的，可以通过止损来取消下跌的